# Predicción de Churn - Telco Customer

Análisis y modelado predictivo para la deserción de clientes de telecomunicaciones.

In [46]:
# === Imports ===
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.metrics import (classification_report, confusion_matrix, precision_score, 
                            accuracy_score, recall_score, f1_score, roc_auc_score, 
                            log_loss, roc_curve, classification_report)
import warnings
warnings.filterwarnings('ignore')


print("Imports cargados correctamente.")

Imports cargados correctamente.


In [47]:
# === Importar el dataset ===
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')


In [48]:
# === Preprocesamiento ===

# 1. Eliminar customerID (no aporta información predictiva)
df = df.drop('customerID', axis=1)

# 2. Convertir TotalCharges a numérico (tiene valores vacíos como strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3. Verificar y eliminar nulos
nulos = df.isnull().sum()
print(f"Valores nulos encontrados:")
print(nulos[nulos > 0])
print(f"\nTotal filas antes: {len(df)}")
df = df.dropna()
print(f"Total filas después: {len(df)}")

# 4. Codificar variable objetivo
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 5. Codificar variables categóricas con Label Encoding
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColumnas categóricas a codificar ({len(categorical_cols)}): {categorical_cols}")

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print(f"\nDataset preprocesado: {df.shape}")
df.head()

Valores nulos encontrados:
TotalCharges    11
dtype: int64

Total filas antes: 7043
Total filas después: 7032

Columnas categóricas a codificar (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Dataset preprocesado: (7032, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [ ]:
def evaluar_y_registrar(model, model_name, best_params, X_tr, X_te, y_tr, y_te, color, cv_strategy):
    """
    Evalúa un modelo, calcula métricas, genera artefactos y los registra en MLflow.
    """
    # Crear carpeta para artefactos si no existe
    os.makedirs("artifacts", exist_ok=True)
    
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    # Metricas de rendimiento
    matriz       = confusion_matrix(y_te, y_pred)
    precision    = precision_score(y_te, y_pred)
    exactitud    = accuracy_score(y_te, y_pred)
    sensibilidad = recall_score(y_te, y_pred)
    puntaje_f1   = f1_score(y_te, y_pred)
    auc_val      = roc_auc_score(y_te, y_proba)
    logloss      = log_loss(y_te, y_proba)
    Classification_Report = classification_report(y_te, y_pred)


    # Cross-validation con RepeatedStratifiedKFold
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv_strategy, scoring='f1')
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()

    tn, fp, fn, tp = matriz.ravel()

    # Imprimir metricas
    print(f"\n  {model_name} - Metricas de Rendimiento:")
    print(f"  Matriz de confusion:")
    print(f"    {matriz}")
    print(f"    TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  Precision:    {precision:.4f}")
    print(f"  Exactitud:    {exactitud:.4f}")
    print(f"  Sensibilidad: {sensibilidad:.4f}")
    print(f"  Puntaje F1:   {puntaje_f1:.4f}")
    print(f"  AUC-ROC:      {auc_val:.4f}")
    print(f"  Log Loss:     {logloss:.4f}")
    print(f"  CV F1 (RSK):  {cv_mean:.4f} (+/-{cv_std:.4f})")
    print(f" Reporte de clasificacion {Classification_Report}")

    # --- Registro en MLflow ---
    mlflow.log_param("model_type", model_name)
    mlflow.log_param("test_size", TEST_SIZE)     # Asegúrate de tener TEST_SIZE en tu entorno
    mlflow.log_param("random_state", RANDOM_STATE) # Asegúrate de tener RANDOM_STATE
    mlflow.log_param("cv_strategy", f"RepeatedStratifiedKFold({CV_FOLDS}x{CV_REPEATS})")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("n_features", X_tr.shape[1])
    mlflow.log_param("n_train", X_tr.shape[0])
    mlflow.log_param("n_test", X_te.shape[0])
    for k, v in best_params.items():
        # Agregamos el prefijo 'model_' para evitar chocar con variables globales
        mlflow.log_param(f"model_{k}", v)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("exactitud", exactitud)
    mlflow.log_metric("sensibilidad", sensibilidad)
    mlflow.log_metric("f1_score", puntaje_f1)
    mlflow.log_metric("auc_roc", auc_val)
    mlflow.log_metric("log_loss", logloss)
    mlflow.log_metric("cv_f1_mean", cv_mean)
    mlflow.log_metric("cv_f1_std", cv_std)
    mlflow.log_metric("TN", int(tn))
    mlflow.log_metric("FP", int(fp))
    mlflow.log_metric("FN", int(fn))
    mlflow.log_metric("TP", int(tp))

    # Artefacto: Matriz de Confusion (Heatmap)
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Retenido (0)', 'Churn (1)'],
                yticklabels=['Retenido (0)', 'Churn (1)'], ax=ax_cm,
                annot_kws={'size': 20}, linewidths=2, linecolor='white')
    ax_cm.set_xlabel('Predicción', fontsize=13)
    ax_cm.set_ylabel('Valor Real', fontsize=13)
    ax_cm.set_title(f'Matriz de Confusión - {model_name}\n'
                    f'Exactitud={exactitud:.4f}', fontsize=14, fontweight='bold')
    ax_cm.text(0.5, -0.12,
               f'TN={tn}  FP={fp}  FN={fn}  TP={tp}',
               transform=ax_cm.transAxes, ha='center', fontsize=11, color='#555')
    fig_cm.tight_layout()
    cm_path = f"artifacts/confusion_matrix_{model_name}.png"
    fig_cm.savefig(cm_path, dpi=150, bbox_inches='tight')
    mlflow.log_artifact(cm_path)
    plt.close(fig_cm)

    # Artefacto: Curva ROC + Youden
    fpr, tpr, thresholds = roc_curve(y_te, y_proba)
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.fill_between(fpr, tpr, alpha=0.15, color=color)
    ax_roc.plot(fpr, tpr, color=color, lw=2.5,
                label=f'Curva ROC (AUC = {auc_val:.4f})')
    ax_roc.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5,
                label='Aleatorio (AUC = 0.50)')
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    ax_roc.scatter(fpr[best_idx], tpr[best_idx],
                   color='#FF5722', s=150, zorder=5,
                   edgecolors='black', lw=1.5,
                   label=f'Punto óptimo (umbral={thresholds[best_idx]:.3f})')
    ax_roc.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
    ax_roc.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
    ax_roc.set_title(f'Curva ROC - {model_name}\nAUC = {auc_val:.4f}',
                     fontsize=14, fontweight='bold')
    ax_roc.legend(loc='lower right', fontsize=10, framealpha=0.9)
    ax_roc.grid(True, alpha=0.3)
    fig_roc.tight_layout()
    roc_path = f"artifacts/roc_curve_{model_name}.png"
    fig_roc.savefig(roc_path, dpi=150, bbox_inches='tight')
    mlflow.log_artifact(roc_path)
    plt.close(fig_roc)

    # Artefacto: Classification Report
    report = classification_report(y_te, y_pred, target_names=['Retenido', 'Churn'])
    report_path = f"artifacts/report_{model_name}.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"{'='*60}\n")
        f.write(f"Classification Report - {model_name}\n")
        f.write(f"Best params: {best_params}\n")
        f.write(f"{'='*60}\n\n")
        f.write(report)
        f.write(f"\nMatriz de Confusion:\n{matriz}\n")
        f.write(f"AUC-ROC: {auc_val:.4f}\n")
    mlflow.log_artifact(report_path)

    # Modelo serializado 
    signature = infer_signature(X_tr, model.predict(X_tr))
    mlflow.sklearn.log_model(
        model, artifact_path="model",
        signature=signature, input_example=X_te[:2]
    )

    mlflow.set_tag("model_type", model_name)
    mlflow.set_tag("dataset", "telco_churn")
    mlflow.set_tag("tuning", "GridSearchCV")

    return {
        "model_name": model_name,
        "best_params": str(best_params),
        "precision": round(precision, 4),
        "exactitud": round(exactitud, 4),
        "sensibilidad": round(sensibilidad, 4),
        "f1_score": round(puntaje_f1, 4),
        "auc_roc": round(auc_val, 4),
        "log_loss": round(logloss, 4),
        "cv_mean": round(cv_mean, 4),
        "cv_std": round(cv_std, 4),
        "fpr": fpr,
        "tpr": tpr,
        "color": color,
        "trained_model": model,
        "confusion_matrix": matriz,
        "y_pred": y_pred,
    }

print("Función evaluar_y_registrar() unificada para Churn definida exitosamente.")


Función evaluar_y_registrar() unificada para Churn definida exitosamente.


In [50]:
# === Separar features y target ===
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features (X): {X.shape}")
print(f"Target (y):   {y.shape}")
print(f"\nDistribución de clases:")
print(y.value_counts())
print(f"\nProporción Churn: {y.mean():.2%}")

cv_strategy = RepeatedStratifiedKFold(
    n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE
)

Features (X): (7032, 19)
Target (y):   (7032,)

Distribución de clases:
Churn
0    5163
1    1869
Name: count, dtype: int64

Proporción Churn: 26.58%


In [54]:
# === División de datos y escalado ===

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv_strategy = RepeatedStratifiedKFold(
    n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE
)

print(f"División de datos:")
print(f"  Train: {X_train_scaled.shape[0]} muestras")
print(f"  Test:  {X_test_scaled.shape[0]} muestras")
print(f"  CV:    RepeatedStratifiedKFold ({CV_FOLDS}x{CV_REPEATS} = {CV_FOLDS*CV_REPEATS} evaluaciones)")
print(f"  Escalado: StandardScaler")


  Decision_Tree - Metricas de Rendimiento:
  Matriz de confusion:
    [[1249  300]
 [ 283  278]]
    TN=1249  FP=300  FN=283  TP=278
  Precision:    0.4810
  Exactitud:    0.7237
  Sensibilidad: 0.4955
  Puntaje F1:   0.4881
  AUC-ROC:      0.6503
  Log Loss:     9.9600
  CV F1 (RSK):  0.4925 (+/-0.0205)


2026/03/28 10:35:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 10:35:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Reporte de Clasificación:
              precision    recall  f1-score   support

Retenido (0)       0.82      0.81      0.81      1549
   Churn (1)       0.48      0.50      0.49       561

    accuracy                           0.72      2110
   macro avg       0.65      0.65      0.65      2110
weighted avg       0.73      0.72      0.72      2110

------------------------------------------------------------
  Run registrado exitosamente en MLflow con ID: 29a3d313ec8649d6ad2f6d0fcafd6a9c


In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
model_name = "Decision_Tree"
color = "#2CA02C" 
model = DecisionTreeClassifier(class_weight='balanced')
model.fit(X_train, y_train)

run_name = f"Evaluacion_{model_name}"
with mlflow.start_run(run_name=run_name) as run:

    parametros_actuales = model.get_params()
    
    result = evaluar_y_registrar(
        model=model,                    
        model_name=model_name,
        best_params=parametros_actuales,
        X_tr=X_train,                 
        X_te=X_test,                   
        y_tr=y_train, 
        y_te=y_test,
        color=color,                    
        cv_strategy=cv_strategy       
    )
    
    print("\n  Reporte de Clasificación:")
    print(classification_report(y_test, result['y_pred'], target_names=['Retenido (0)', 'Churn (1)']))
    print("-" * 60)

    
    print(f"  Run registrado exitosamente en MLflow con ID: {run.info.run_id}")


  Decision_Tree - Metricas de Rendimiento:
  Matriz de confusion:
    [[1250  299]
 [ 286  275]]
    TN=1250  FP=299  FN=286  TP=275
  Precision:    0.4791
  Exactitud:    0.7227
  Sensibilidad: 0.4902
  Puntaje F1:   0.4846
  AUC-ROC:      0.6487
  Log Loss:     9.9438
  CV F1 (RSK):  0.5037 (+/-0.0128)


2026/03/28 10:37:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 10:37:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Reporte de Clasificación:
              precision    recall  f1-score   support

Retenido (0)       0.81      0.81      0.81      1549
   Churn (1)       0.48      0.49      0.48       561

    accuracy                           0.72      2110
   macro avg       0.65      0.65      0.65      2110
weighted avg       0.72      0.72      0.72      2110

------------------------------------------------------------
  Run registrado exitosamente en MLflow con ID: c2bf1d9a57a44f2698df59d53f099f7a


In [ ]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train, y_train)

print(f"Distribución tras OverSampling:\n{pd.Series(y_res).value_counts(normalize=True)}")
print("-" * 30)

# --- PASO E: Entrenamiento y Evaluación ---
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_res, y_res)

y_pred = model.predict(X_test)

print("REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred))

Distribución tras OverSampling:
Churn
1    0.5
0    0.5
Name: proportion, dtype: float64
------------------------------
REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1549
           1       0.50      0.76      0.60       561

    accuracy                           0.73      2110
   macro avg       0.70      0.74      0.70      2110
weighted avg       0.79      0.73      0.75      2110

